# Traffic Demand Prediction V6


In [1]:
"""V6: Combine day-48 lag features (from 90-score solution) + LightGBM ensemble + lat/lon + tuned hyperparams."""import time, json, zipfile, warningsimport numpy as npimport pandas as pdfrom sklearn.model_selection import KFoldfrom sklearn.ensemble import HistGradientBoostingRegressorfrom sklearn.metrics import r2_scorefrom sklearn.preprocessing import OrdinalEncoder, LabelEncoderimport lightgbm as lgbwarnings.filterwarnings('ignore')t0 = time.time()RNG = 42TARGET = "demand"train = pd.read_csv("dataset/train.csv")test = pd.read_csv("dataset/test.csv")print(f"train {train.shape}  test {test.shape}")# ==================== GEOHASH DECODE ====================BASE32 = '0123456789bcdefghjkmnpqrstuvwxyz'def decode_geohash(gh):    bits = []    for c in gh:        bits.extend([int(b) for b in format(BASE32.index(c), '05b')])    def tc(bl, lo, hi):        for b in bl:            mid = (lo+hi)/2            if b: lo = mid            else: hi = mid        return (lo+hi)/2    return tc(bits[1::2], -90, 90), tc(bits[0::2], -180, 180)all_geos = set(train['geohash'].unique()) | set(test['geohash'].unique())geo_coords = {g: decode_geohash(g) for g in all_geos}# ==================== BASE FEATURES ====================def fe(df):    df = df.copy()    ts = df["timestamp"].str.split(":", expand=True).astype(int)    df["minute"] = ts[0] * 60 + ts[1]    df["hour"] = ts[0]    df["minute_of_hour"] = ts[1]    df["time_slot"] = df["minute"] // 15    df["min_sin"] = np.sin(2 * np.pi * df["minute"] / 1440)    df["min_cos"] = np.cos(2 * np.pi * df["minute"] / 1440)    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)    df["slot_sin"] = np.sin(2 * np.pi * df["time_slot"] / 96)    df["slot_cos"] = np.cos(2 * np.pi * df["time_slot"] / 96)    df["dow"] = df["day"] % 7    df["is_peak"] = ((df["hour"] >= 7) & (df["hour"] <= 13)).astype(int)    for n in (3, 4, 5):        df[f"gh{n}"] = df["geohash"].str[:n]    coords = df["geohash"].map(geo_coords)    df["lat"] = coords.apply(lambda x: x[0])    df["lon"] = coords.apply(lambda x: x[1])    df["LV"] = (df["LargeVehicles"] == "Allowed").astype(int)    df["LM"] = (df["Landmarks"] == "Yes").astype(int)    road_map = {"Residential": 0, "Street": 1, "Highway": 2}    df["RT_ord"] = df["RoadType"].map(road_map).fillna(0).astype(int)    df["is_hw"] = (df["RoadType"] == "Highway").astype(int)    df["is_st"] = (df["RoadType"] == "Street").astype(int)    df["hc"] = ((df["NumberofLanes"] >= 4) | (df["is_hw"] == 1)).astype(int)    df["lanes_hw"] = df["NumberofLanes"] * df["is_hw"]    df["temp_missing"] = df["Temperature"].isnull().astype(int)    return dftrain_fe = fe(train)test_fe = fe(test)# ==================== DAY-48 LAG FEATURES ====================d48 = train_fe[train_fe["day"] == 48][["geohash", "hour", "minute", "time_slot", TARGET]].copy()# Exact same-timestamp lag: (geohash, minute-of-day)lag_exact = d48.groupby(["geohash", "minute"])[TARGET].mean().rename("lag_same_ts").reset_index()# Same-hour lag: (geohash, hour)lag_hour = d48.groupby(["geohash", "hour"])[TARGET].mean().rename("lag_same_hour").reset_index()# Same time-slot lag: (geohash, time_slot)lag_slot = d48.groupby(["geohash", "time_slot"])[TARGET].mean().rename("lag_same_slot").reset_index()# Per-geohash day-48 aggregatesgh_aggs = d48.groupby("geohash")[TARGET].agg(    gh_d48_mean="mean", gh_d48_std="std", gh_d48_med="median",    gh_d48_min="min", gh_d48_max="max",    gh_d48_q25=lambda s: s.quantile(0.25),    gh_d48_q75=lambda s: s.quantile(0.75),    gh_d48_n="count").reset_index()gh_aggs["gh_d48_std"] = gh_aggs["gh_d48_std"].fillna(0)gh_aggs["gh_d48_range"] = gh_aggs["gh_d48_max"] - gh_aggs["gh_d48_min"]gh_aggs["gh_d48_iqr"] = gh_aggs["gh_d48_q75"] - gh_aggs["gh_d48_q25"]# Per-geohash, per-hour day-48 aggregatesgh_hour_aggs = d48.groupby(["geohash", "hour"])[TARGET].agg(    gh_d48_hr_mean="mean", gh_d48_hr_std="std", gh_d48_hr_n="count").reset_index()gh_hour_aggs["gh_d48_hr_std"] = gh_hour_aggs["gh_d48_hr_std"].fillna(0)def attach_lags(df):    df = df.merge(lag_exact, on=["geohash", "minute"], how="left")    df = df.merge(lag_hour, on=["geohash", "hour"], how="left")    df = df.merge(lag_slot, on=["geohash", "time_slot"], how="left")    df = df.merge(gh_aggs, on=["geohash"], how="left")    df = df.merge(gh_hour_aggs, on=["geohash", "hour"], how="left")    return dftrain_fe = attach_lags(train_fe)test_fe = attach_lags(test_fe)# For day-48 train rows, lag features leak (they're from the same day) -> set to NaNday48_mask = train_fe["day"] == 48lag_cols_to_mask = ["lag_same_ts", "lag_same_hour", "lag_same_slot",                    "gh_d48_mean", "gh_d48_std", "gh_d48_med", "gh_d48_min", "gh_d48_max",                    "gh_d48_q25", "gh_d48_q75", "gh_d48_n", "gh_d48_range", "gh_d48_iqr",                    "gh_d48_hr_mean", "gh_d48_hr_std", "gh_d48_hr_n"]for c in lag_cols_to_mask:    train_fe.loc[day48_mask, c] = np.nanprint(f"lag coverage test: exact={test_fe['lag_same_ts'].notna().mean():.3f} "      f"hour={test_fe['lag_same_hour'].notna().mean():.3f} "      f"gh_mean={test_fe['gh_d48_mean'].notna().mean():.3f}")# ==================== DAY-49 LOO FEATURES ====================d49_train = train_fe[train_fe["day"] == 49]d49_sum = d49_train.groupby("geohash")[TARGET].sum()d49_count = d49_train.groupby("geohash")[TARGET].count()# LOO mean for day-49 train rowsloo_mean = pd.Series(np.nan, index=train_fe.index)gh_arr = train_fe["geohash"].valuesy_arr = train_fe[TARGET].valuesmask49 = (~day48_mask).valuesfor i in np.where(mask49)[0]:    g = gh_arr[i]    n = d49_count.get(g, 0)    if n > 1:        loo_mean.iloc[i] = (d49_sum[g] - y_arr[i]) / (n - 1)train_fe["gh_d49_mean_loo"] = loo_meantest_fe["gh_d49_mean_loo"] = (d49_sum / d49_count).reindex(test_fe["geohash"]).values# Day-49 hour-level LOOd49_hr_sum = d49_train.groupby(["geohash", "hour"])[TARGET].sum()d49_hr_count = d49_train.groupby(["geohash", "hour"])[TARGET].count()loo_hr_mean = pd.Series(np.nan, index=train_fe.index)hr_arr = train_fe["hour"].valuesfor i in np.where(mask49)[0]:    g, h = gh_arr[i], hr_arr[i]    key = (g, h)    if key in d49_hr_count:        n = d49_hr_count[key]        if n > 1:            loo_hr_mean.iloc[i] = (d49_hr_sum[key] - y_arr[i]) / (n - 1)train_fe["gh_d49_hr_mean_loo"] = loo_hr_meand49_hr_full = (d49_hr_sum / d49_hr_count).reset_index()d49_hr_full.columns = ["geohash", "hour", "gh_d49_hr_mean_loo"]test_fe = test_fe.drop(columns=["gh_d49_hr_mean_loo"], errors="ignore")test_fe = test_fe.merge(d49_hr_full, on=["geohash", "hour"], how="left")# ==================== OOF TARGET ENCODING ====================def target_encode_oof(train_df, test_df, col, y, n_splits=5, smoothing=20, seed=RNG):    gm = y.mean()    oof = np.zeros(len(train_df), dtype=np.float32)    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)    for tr, va in kf.split(train_df):        agg = pd.DataFrame({col: train_df[col].iloc[tr].values, "y": y[tr]}).groupby(col)["y"].agg(["mean", "count"])        sm = (agg["mean"] * agg["count"] + gm * smoothing) / (agg["count"] + smoothing)        oof[va] = train_df[col].iloc[va].map(sm.to_dict()).fillna(gm).values    agg = pd.DataFrame({col: train_df[col].values, "y": y}).groupby(col)["y"].agg(["mean", "count"])    sm = (agg["mean"] * agg["count"] + gm * smoothing) / (agg["count"] + smoothing)    test_enc = test_df[col].map(sm.to_dict()).fillna(gm).values.astype(np.float32)    return oof, test_ency_raw = train_fe[TARGET].astype(np.float32).valuesfor c in ["geohash"]:    oof, te = target_encode_oof(train_fe, test_fe, c, y_raw, smoothing=20)    train_fe[f"{c}_te"] = oof    test_fe[f"{c}_te"] = te# ==================== CATEGORICAL ENCODING ====================cat_cols = ["gh3", "gh4", "gh5", "RoadType", "LargeVehicles", "Landmarks", "Weather"]for c in cat_cols:    train_fe[c] = train_fe[c].fillna("MISSING").astype(str)    test_fe[c] = test_fe[c].fillna("MISSING").astype(str)temp_med = train_fe["Temperature"].median()train_fe["Temperature"] = train_fe["Temperature"].fillna(temp_med)test_fe["Temperature"] = test_fe["Temperature"].fillna(temp_med)enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)all_cat = pd.concat([train_fe[cat_cols], test_fe[cat_cols]], axis=0)enc.fit(all_cat)train_fe[cat_cols] = enc.transform(train_fe[cat_cols])test_fe[cat_cols] = enc.transform(test_fe[cat_cols])# Geohash label encoding for LGBle_gh = LabelEncoder()le_gh.fit(list(all_geos))train_fe["geo_le"] = le_gh.transform(train_fe["geohash"])test_fe["geo_le"] = le_gh.transform(test_fe["geohash"])# ==================== FEATURE SELECTION ====================num_cols = ["day", "minute", "hour", "minute_of_hour", "time_slot",            "min_sin", "min_cos", "hour_sin", "hour_cos", "slot_sin", "slot_cos",            "dow", "is_peak", "NumberofLanes", "Temperature",            "lat", "lon", "LV", "LM", "RT_ord", "is_hw", "is_st", "hc",            "lanes_hw", "temp_missing", "geo_le"]lag_cols = ["lag_same_ts", "lag_same_hour", "lag_same_slot",            "gh_d48_mean", "gh_d48_std", "gh_d48_med", "gh_d48_min", "gh_d48_max",            "gh_d48_q25", "gh_d48_q75", "gh_d48_n", "gh_d48_range", "gh_d48_iqr",            "gh_d48_hr_mean", "gh_d48_hr_std", "gh_d48_hr_n",            "gh_d49_mean_loo", "gh_d49_hr_mean_loo", "geohash_te"]features = cat_cols + num_cols + lag_colscat_idx = [features.index(c) for c in cat_cols]X = train_fe[features].astype(np.float32)y = y_rawX_test = test_fe[features].astype(np.float32)print(f"\nfeatures: {len(features)}  cat: {len(cat_cols)}  num: {len(num_cols)}  lag: {len(lag_cols)}")print(f"X={X.shape}  X_test={X_test.shape}")print(f"setup elapsed: {time.time()-t0:.1f}s")# ==================== TEMPORAL VALIDATION ====================d48_idx = train_fe[train_fe["day"] == 48].indexd49_idx = train_fe[train_fe["day"] == 49].indexprint("\n=== Temporal Validation (d48 -> d49) ===")# HGBhgb_temp = HistGradientBoostingRegressor(    max_iter=2500, learning_rate=0.02, max_leaf_nodes=255,    min_samples_leaf=10, l2_regularization=1.0,    categorical_features=cat_idx, random_state=RNG,    early_stopping=True, validation_fraction=0.1, n_iter_no_change=80)hgb_temp.fit(X.loc[d48_idx], y[d48_idx])hgb_temp_pred = hgb_temp.predict(X.loc[d49_idx])hgb_temp_r2 = r2_score(y[d49_idx], hgb_temp_pred)print(f"  HGB temporal R2: {hgb_temp_r2:.6f} (score: {max(0,100*hgb_temp_r2):.2f})")# LGB (features without cat_cols for LGB, or handle differently)lgb_features = num_cols + lag_cols  # LGB handles NaN nativelyX_lgb = train_fe[lgb_features].astype(np.float32)X_test_lgb = test_fe[lgb_features].astype(np.float32)lgb_temp = lgb.LGBMRegressor(    n_estimators=3000, learning_rate=0.02, num_leaves=63,    min_child_samples=30, feature_fraction=0.6, bagging_fraction=0.7, bagging_freq=5,    reg_alpha=1.0, reg_lambda=5.0, verbose=-1, random_state=RNG)lgb_temp.fit(X_lgb.loc[d48_idx], y[d48_idx],             eval_set=[(X_lgb.loc[d49_idx], y[d49_idx])],             callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])lgb_temp_pred = lgb_temp.predict(X_lgb.loc[d49_idx])lgb_temp_r2 = r2_score(y[d49_idx], lgb_temp_pred)print(f"  LGB temporal R2: {lgb_temp_r2:.6f} (score: {max(0,100*lgb_temp_r2):.2f})")# Blendfor alpha in [0.3, 0.4, 0.5, 0.6, 0.7]:    bl = alpha * hgb_temp_pred + (1-alpha) * lgb_temp_pred    r2 = r2_score(y[d49_idx], bl)    print(f"  Blend a={alpha}: R2={r2:.6f}")# ==================== FULL CV ====================N_FOLDS = 5kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RNG)# HGB CVoof_hgb = np.zeros(len(y), dtype=np.float32)test_hgb = np.zeros(len(X_test), dtype=np.float64)print(f"\n=== HGB {N_FOLDS}-Fold CV ===")for fold, (tr, va) in enumerate(kf.split(X)):    m = HistGradientBoostingRegressor(        max_iter=2500, learning_rate=0.02, max_leaf_nodes=255,        min_samples_leaf=10, l2_regularization=1.0,        categorical_features=cat_idx, random_state=RNG,        early_stopping=True, validation_fraction=0.1, n_iter_no_change=80)    m.fit(X.iloc[tr], y[tr])    oof_hgb[va] = m.predict(X.iloc[va])    test_hgb += m.predict(X_test) / N_FOLDS    print(f"  Fold {fold+1}: R2={r2_score(y[va], oof_hgb[va]):.5f} iters={m.n_iter_}")hgb_oof_r2 = r2_score(y, oof_hgb)d49_only = (train_fe["day"] == 49).valuesprint(f"  OOF R2={hgb_oof_r2:.5f}  d49-only={r2_score(y[d49_only], oof_hgb[d49_only]):.5f}")# LGB CVoof_lgb = np.zeros(len(y), dtype=np.float32)test_lgb_cv = np.zeros(len(X_test_lgb), dtype=np.float64)print(f"\n=== LGB {N_FOLDS}-Fold CV ===")for fold, (tr, va) in enumerate(kf.split(X_lgb)):    m = lgb.LGBMRegressor(        n_estimators=3000, learning_rate=0.02, num_leaves=63,        min_child_samples=30, feature_fraction=0.6, bagging_fraction=0.7, bagging_freq=5,        reg_alpha=1.0, reg_lambda=5.0, verbose=-1, random_state=RNG)    m.fit(X_lgb.iloc[tr], y[tr], eval_set=[(X_lgb.iloc[va], y[va])],          callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])    oof_lgb[va] = m.predict(X_lgb.iloc[va])    test_lgb_cv += m.predict(X_test_lgb) / N_FOLDS    print(f"  Fold {fold+1}: R2={r2_score(y[va], oof_lgb[va]):.5f} iters={m.best_iteration_}")lgb_oof_r2 = r2_score(y, oof_lgb)print(f"  OOF R2={lgb_oof_r2:.5f}  d49-only={r2_score(y[d49_only], oof_lgb[d49_only]):.5f}")# ==================== MULTI-SEED LGB ====================seeds = [0, 7, 42, 99, 123, 256, 314, 456, 789, 2024]test_lgb_seeds = np.zeros(len(X_test_lgb), dtype=np.float64)print(f"\n=== LGB {len(seeds)}-seed ensemble (full train) ===")for seed in seeds:    m = lgb.LGBMRegressor(        n_estimators=2000, learning_rate=0.02, num_leaves=63,        min_child_samples=30, feature_fraction=0.6, bagging_fraction=0.7, bagging_freq=5,        reg_alpha=1.0, reg_lambda=5.0, verbose=-1, random_state=seed)    m.fit(X_lgb, y)    test_lgb_seeds += m.predict(X_test_lgb) / len(seeds)    print(f"  Seed {seed} done")# ==================== MULTI-SEED HGB ====================test_hgb_seeds = np.zeros(len(X_test), dtype=np.float64)hgb_seeds = [42, 123, 456]print(f"\n=== HGB {len(hgb_seeds)}-seed ensemble (full train) ===")for seed in hgb_seeds:    m = HistGradientBoostingRegressor(        max_iter=2500, learning_rate=0.02, max_leaf_nodes=255,        min_samples_leaf=10, l2_regularization=1.0,        categorical_features=cat_idx, random_state=seed,        early_stopping=True, validation_fraction=0.1, n_iter_no_change=80)    m.fit(X, y)    test_hgb_seeds += m.predict(X_test) / len(hgb_seeds)    print(f"  Seed {seed} done (iters={m.n_iter_})")# ==================== OPTIMAL BLEND ====================# Find best OOF blend of HGB and LGBbest_r2 = -1best_alpha = 0.5for alpha in np.arange(0, 1.01, 0.05):    bl = alpha * oof_hgb + (1 - alpha) * oof_lgb    r2 = r2_score(y, bl)    if r2 > best_r2:        best_r2 = r2        best_alpha = alphaprint(f"\nBest OOF blend: alpha(HGB)={best_alpha:.2f} R2={best_r2:.6f}")# Final test prediction: blend CV + seed ensemblestest_hgb_final = 0.5 * test_hgb + 0.5 * test_hgb_seedstest_lgb_final = 0.5 * test_lgb_cv + 0.5 * test_lgb_seedsfinal_preds = best_alpha * test_hgb_final + (1 - best_alpha) * test_lgb_finalfinal_preds = np.clip(final_preds, 0, 1)print(f"Final preds: mean={final_preds.mean():.5f} std={final_preds.std():.5f}")# ==================== SUBMISSION ====================sub = pd.DataFrame({"Index": test_fe["Index"].values, "demand": final_preds})sub.to_csv("predictions.csv", index=False)print(f"\npredictions.csv saved: {len(sub)} rows")# Feature importanceimp = pd.DataFrame({"feature": lgb_features, "importance": m.feature_importances_})print("\nTop 20 LGB features:")print(imp.sort_values("importance", ascending=False).head(20).to_string(index=False))# ==================== APPROACH + NOTEBOOK + ZIP ====================approach = f"""Traffic Demand Prediction - V6 Final ApproachKey Insight: Train=day48 (69K rows) + day49 early hours (8K rows). Test=day49 hours 2-13.The most powerful features are DAY-48 LAG features: what demand looked like at the samelocation/time yesterday.1. Feature Engineering ({len(features)} features):   a) Day-48 Lag Features (key innovation):      - Exact same-timestamp lag: demand at (geohash, minute) on day 48      - Same-hour lag: mean demand at (geohash, hour) on day 48      - Same time-slot lag: mean demand at (geohash, 15min-slot) on day 48      - Per-geohash day-48 aggregates: mean/std/median/min/max/q25/q75/count/range/IQR      - Per-geohash-hour day-48 aggregates: mean/std/count      - Properly masked to NaN for day-48 train rows to prevent leakage   b) Day-49 LOO Features:      - Leave-one-out geohash mean from day-49 train rows      - Leave-one-out geohash-hour mean from day-49 train rows   c) OOF Target Encoding: geohash with smoothing=20   d) Geohash decoded to lat/lon coordinates   e) Cyclical time encoding (minute, hour, slot)   f) Categorical ordinal encoding (gh3/4/5, RoadType, Weather, etc.)   g) Road/vehicle interaction features2. Modeling:   - HGB: max_leaf_nodes=255, lr=0.02, native categorical support   - LGB: 63 leaves, lr=0.02, reg_alpha=1.0, reg_lambda=5.0   - {N_FOLDS}-fold CV for both models   - Multi-seed ensemble: {len(seeds)} LGB seeds + {len(hgb_seeds)} HGB seeds   - Optimal OOF blend: alpha(HGB)={best_alpha:.2f}   - OOF R2: {best_r2:.6f}3. Tools: Python, pandas, numpy, LightGBM, scikit-learn"""with open("approach.txt", "w") as f:    f.write(approach.strip())with open("solution_v6.py", "r") as f:    src = f.read()nb = {    "cells": [        {"cell_type": "markdown", "metadata": {}, "source": ["# Traffic Demand Prediction V6\n"]},        {"cell_type": "code", "execution_count": 1, "metadata": {}, "outputs": [], "source": src.split("\n")}    ],    "metadata": {"kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},                 "language_info": {"name": "python", "version": "3.8.0"}},    "nbformat": 4, "nbformat_minor": 4}with open("solution.ipynb", "w") as f:    json.dump(nb, f, indent=1)with zipfile.ZipFile("submission.zip", "w") as z:    z.write("predictions.csv")    z.write("approach.txt")    z.write("solution.ipynb")print(f"\nAll files ready! Total elapsed: {time.time()-t0:.1f}s")